# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview

**TODO — obserwacje (2–4 zdania):** które kolumny mają nieoczekiwany typ? Gdzie są braki?
Które kolumny liczbowe są w rzeczywistości kategoriami?

mamy 3150 rekordów i 14 kolumn, Nie ma zadnych braków. Age group i charge amount bierzemy jako cechy porządkowe, Kolumny Complains Tariff Plan i status sa liczbowe a reprezentoja kategorie

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()

**TODO — metryka główna (2–3 zdania):** którą metrykę (accuracy, precision, recall, F1)
wybieracie jako główną i dlaczego, biorąc pod uwagę udział klasy pozytywnej i koszt błędów
w waszym problemie? To samo uzasadnienie wpiszcie do karty projektu w README.

Wybieramy F1, bedzie ona najlepsza ponieważ zbiór danych jest niezbalansowany 15% klientów nalezy do klasy pozytywnej, Accuracy jest błedne bo model przewidojac tylko negatywne wartosci osiągnał by juz ok 84% skutecznosci. f1 uwzględnia precision i recall dlatego pozwala on realnie ocenic skutecznosc wykrywania klientów zagrozonych odejsciem

## 5. EDA

**TODO:** co najmniej dwie analizy — jedna dla cechy numerycznej, jedna dla kategorycznej.
Wybierajcie wyłącznie spośród ALLOWED FEATURES.

In [ ]:
import matplotlib.pyplot as plt

print(df_raw.groupby("Churn")["Seconds of Use"].describe())

df_raw.boxplot(column="Seconds of Use", by="Churn")
plt.suptitle("")
plt.title("Seconds of Use a Churn")
plt.ylabel("Seconds of Use")
plt.show()

In [ ]:
udzial = pd.crosstab(df_raw["Complains"], df_raw["Churn"], normalize="index")

print(udzial)

udzial.plot(kind="bar")
plt.title("Complains a Churn")
plt.xlabel("Complains")
plt.ylabel("Udział klientów")
plt.show()

Nie dodajemy nowych reguł. Wszystkie kolumny są liczbowe i nie ma braków



1. 1566 sekund srednio kozystali z programu klienci którzy odeszli a pozostali 5014 s, czyli mniejsza aktywnosc przedłada sie na większe ryzyko odejscia
2.z klientów którzy zgłosili reklamacje aż 83% odeszła, a klienci bez reklamacji maja tylko 10% odejść
Obie cechy zostawiamy w modelu.

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")

**TODO — reguły input normalization:** jakie reguły dopisujecie do `normalize_input()` i dlaczego
są bezpieczne przy predykcji pojedynczego rekordu? Po zmianie funkcji uruchomcie `pytest`.

Nie dodajemy innych reguł normalizacji, poniewaz wszystkie kolumny sa liczbowe i nei wystepuja brakujace wartosci.
normalize_input() zachowuje wartosci liczbowe i czysci tekst, Nie bierze targetu ani statystyk całego zbioru co czyni ja bezpieczna podczas predykcji pojedynczych wartosci


In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")

**TODO — decyzja o training-only filtering:** czy usuwacie dodatkowo jakieś rekordy
(np. dokładne duplikaty)? Dlaczego tak albo dlaczego nie?

Usuwamy 300 dokładnych duplikatów poniewaz nie wnosza one nowych danych i moga zniekształcic wynik.
po wykonaniu filtrowanai pozostaje nam 2850 rekórdów z 3150, robimy to przed podziałem train test. filtrujemy tylko podczas przygotowania danych a nie podczas predykcji

## 8. Typy cech → `project.yaml`

**TODO:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

**TODO — w 3–5 punktach:** decyzje o input normalization, training-only filtering, typach cech
i metryce głównej. Te decyzje będą punktem wyjścia do ADR-1 w Sprincie 2.

1.iranian_churn zsawiera 3150 rekordów i 14 kolum, nie ma brakujących wartosci
2.Training only filtering usuwa 300 duplikatów czyli zostaje nam 2850 rekordów
3.Input normalization nie potrzebuje nowych reguł bo wszystkie kolumny maja typ liczbowy
4.wybralismy 9 cech numerycznych i 3 kategoryczne, cystomer value zostało odrzucone
5.metryka głowna jest f1 bo wartosci 1 stanowaia 15% danych
6.nizszy czas korzystania z usługi i reklamacjie zwiekszaja ryzyko odejscia klienta

In [ ]:
print("Rozmiar danych:", df_raw.shape)

print("\nTypy kolumn:")
print(df_raw.dtypes)

print("\nBrakujące wartości:")
print(df_raw.isna().sum())

print("\nDuplikaty:")
print(df_raw.duplicated().sum())

print("\nRozkład Churn:")
print(df_raw["Churn"].value_counts())

for col in ["Charge Amount", "Age Group", "Complains", "Tariff Plan", "Status"]:
    print(f"\n{col}:")
    print(df_raw[col].value_counts(dropna=False))

In [ ]:
print("Braki łącznie:", df_raw.isna().sum().sum())
print("Duplikaty:", df_raw.duplicated().sum())
print("Churn:", df_raw["Churn"].value_counts().to_dict())
print(
    "Kategorie:",
    {col: sorted(df_raw[col].unique().tolist()) for col in ["Complains", "Tariff Plan", "Status"]},
)




- Zbiór zawiera 3150 rekordów i 14 kolumn.
- Nie wykryto brakujących wartości.
- Wykryto 300 duplikatów pełnych rekordów.
- Target: Churn (0 = pozostaje, 1 = odchodzi).
- Klasa pozytywna stanowi 15,7% danych.
- Complains, Tariff Plan i Status to cechy kategoryczne zakodowane liczbami.
- Charge Amount i Age Group traktujemy jako cechy porządkowe.
- Customer Value wykluczamy zgodnie z kontraktem.
- Podział train/test: stratified_random, 80/20.



1. Nie jest potrzebne uzupełnianie braków w obecnym zbiorze.
2. Duplikatów na razie nie usuwamy automatycznie.
   Ich obsługę należy uzgodnić przed trenowaniem.
3. Kategorie powinny być odpowiednio kodowane
   w sklearn Pipeline.
4. Nie należy używać Customer Value jako cechy.
5. Normalizacja wejścia nie może korzystać z targetu
   ani statystyk całego datasetu.
